# Notebook 02: Transformer Abstractive Summarization Testing
Evaluates BART (`facebook/bart-large-cnn`) and chunking algorithms on long documents with ROUGE-1/2/L and consistency scoring.

In [ ]:
import os
import pandas as pd
from src.summarization import TextSummarizer
from src.evaluation import Evaluator
from src.utils import load_json, calculate_compression_ratio

# Initialize models
summarizer = TextSummarizer(model_name="facebook/bart-large-cnn", use_lazy_loading=True)
evaluator = Evaluator()

samples = load_json("../data/raw/sample_conversations.json")[:5]
print(f"Testing summarizer on {len(samples)} sample documents...")

In [ ]:
results = []
for item in samples:
    text = item['text']
    summary = summarizer.summarize(text)
    rouge = evaluator.calculate_rouge_scores(text, summary)
    consistency = evaluator.consistency_check(text, summary)
    
    results.append({
        "id": item["id"],
        "domain": item["domain"],
        "orig_words": len(text.split()),
        "summary_words": len(summary.split()),
        "compression": calculate_compression_ratio(text, summary),
        "rouge1_f": rouge["rouge1_f"],
        "rouge2_f": rouge["rouge2_f"],
        "rougeL_f": rouge["rougeL_f"],
        "consistency": consistency["consistency_score"]
    })

df_res = pd.DataFrame(results)
df_res